# `vibevoice` with VoiceHub

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/models/vibevoice.ipynb)

- Task: **Text to speech**
- Hugging Face ID: [`microsoft/VibeVoice-Realtime-0.5B`](https://huggingface.co/microsoft/VibeVoice-Realtime-0.5B)

Install VoiceHub using the [installation guide](https://kadirnar.github.io/voicehub/getting-started/installation/)
before opening this model workflow. This notebook contains no package-install cell.

The registry check is safe to run without downloading weights. Inference is disabled by default.


In [ ]:
from pathlib import Path

RUN_INFERENCE = False
MODEL_TYPE = 'vibevoice'
CHECKPOINT = 'microsoft/VibeVoice-Realtime-0.5B'
DEVICE = "cuda"
TEXT = 'VoiceHub keeps model integrations explicit and reproducible.'
OUTPUT_FILE = Path("artifacts/vibevoice.wav")


## Inspect registry support


In [ ]:
from voicehub import get_model_spec

model_spec = get_model_spec(MODEL_TYPE)
assert model_spec.task.value == 'text-to-speech'
assert model_spec.default_model_path == CHECKPOINT
print("task:", model_spec.task.value)
print("checkpoint:", model_spec.default_model_path)
print("capabilities:", ", ".join(model_spec.capabilities))
print("training:", model_spec.training.support.value)


## Run inference

Loads the audited VibeVoice realtime stages without claiming an unverified text-to-waveform loop.

High-level cached-prompt synthesis intentionally fails closed until cache serialization, chunk boundaries, and waveform parity are verified.

This VoiceHub example is maintained in this repository and is not copied from an upstream package snippet. Set `RUN_INFERENCE = True` after reviewing inputs.


In [ ]:
if RUN_INFERENCE:
    from voicehub import AutoModelForTextToSpeech

    model = AutoModelForTextToSpeech.from_pretrained(
        CHECKPOINT,
        model_type=MODEL_TYPE,
        device=DEVICE,
        lazy_load=True,
    )
    model.load()
    required_stages = (
        "forward_lm",
        "forward_tts_lm",
        "sample_speech_latents",
        "decode_speech_latents",
    )
    missing = [name for name in required_stages if not hasattr(model.model, name)]
    if missing:
        raise RuntimeError(f"Missing audited VibeVoice stage(s): {', '.join(missing)}")
    print("High-level synthesis is not verified; available native stages:", required_stages)


## Next

See the [inference guide](https://kadirnar.github.io/voicehub/guides/inference/) and [model catalog](https://kadirnar.github.io/voicehub/models/) for the shared runtime contract and model-specific limitations.
